<a href="https://colab.research.google.com/github/RysanDeluna/start-bioinfo26/blob/main/class-7/DESeq2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Você conhece o Jupyter Notebooks?
O Jupyter Notebooks são um conjunto de ferramentas que combina códigos executáveis, textos com informações, visualizações e outros elementos em um único documento. Amplamente usado em análises de dados, aprendizado de máquina e em análises computacionais, compatível com múltiplas linguagens de programação, sendo o Python a mais popular. Sua interface intuitiva simplifica a exploração dos dados e experimentos, documentados em tempo real.


Aqui, nós temos células de texto e células de código que atendem diferentes propostas para organizar e apresentar o conteúdo dentro dos notebooks.



# Google Colaboratory

O Google Colab é uma plataforma gratuita baseada em nuvem que permite criar, rodar e compartilhar Jupyter notebooks diretamente em seu navegador. O jupyter suporta linguagem como Python e permite acesso a poderosas recursos computacionais como GPUs e TPUs, sendo ideal para relizar tarefas de aprendizado de máquinas e de ciência de dados.


Além disso, é integrado com o Google Drive, permitindo fácil armazenamento e colaborações em tempo real.

#Pipeline DESeq2

## O dataset airway

Esse é um conjunto de dados (dataset) didático do Bioconductor. Ideal para ensino porque é pequeno, limpo e já vem pronto para ser usado com DESeq2.

**Experimento:** algumas amostras foram tratadas com dexametasona (glicocorticoide anti-inflamatório), outras ficaram como controle.

**Formato:** matriz de contagens de RNA-seq (genes x amostras) + metadados (informações sobre condição, tipo celular, lote, etc.).

* O Bioconductor é um projeto de software livre, de código aberto e de desenvolvimento aberto para análise e compreensão de dados genômicos gerados por experimentos em laboratórios na área de biologia molecular.
Origem: células de vias aéreas humanas cultivadas em laboratório.


## 1. Instalando e Carregando Dados

No Google Colab já habilitado para o R. Então o que faremos aqui é instalar e carregar os pacotes necessários para o trabalho.

In [ ]:
# Instalar pacotes necessários
install.packages('BiocManager')
BiocManager::install('DESeq2')
BiocManager::install('EnhancedVolcano')
BiocManager::install("airway")
BiocManager::install("pheatmap")
BiocManager::install("ggplot2")
BiocManager::install("readxl")

# Carregamento das bibliotecas
library(DESeq2)      # Pacote principal para análise de expressão diferencial
library(airway)      # Pacote com os dados do RNA-Seq
library(ggplot2)     # Sistema avançado para criação de gráficos
library(pheatmap)    # Criação de heatmaps (mapas de calor)
library(readxl)      # Leitura de arquivos do Microsoft Excel


## 2. Inspecionando os dados
### O que estamos fazendo aqui
* `airway`: carrega o dataset real de RNA-seq.

* `colData()`: explora os metadados das amostras (condição, tipo celular, etc.).

* `assay()` + `head()`: inspeciona a matriz de contagens de genes.

* `dim()`: verifica quantos genes e amostras existem.

* `colSums()` e `rowSums()`: avalia profundidade de sequenciamento e expressão total por gene.

* `summary()`: dá uma visão geral da distribuição das contagens.

In [ ]:
# Carregar os dados do airway
data("airway")

# Visualizar o objeto airway
airway
# Este comando mostra um resumo do objeto SummarizedExperiment:
# número de genes (linhas), número de amostras (colunas) e metadados associados.


In [ ]:
# Ver metadados das amostras
colData(airway)
# colData() retorna informações sobre cada amostra:
# condição experimental (dex), tipo celular (cell), lote, etc.

In [ ]:
# Ver primeiras linhas da matriz de contagens
head(assay(airway))
# assay() acessa a matriz de contagens (genes x amostras).
# head() mostra apenas o topo (primeiras linhas), útil para inspecionar rapidamente.

In [ ]:
# Dimensões do dataset
dim(airway)
# dim() mostra quantos genes (linhas) e quantas amostras (colunas) existem.


In [ ]:
# Nomes das amostras
colnames(airway)
#colnames() mostra o nome das colunas, que neste caso estão os nomes das amostras


In [ ]:
# Nomes dos genes
rownames(airway)[1:10]
#rownames() são as linhas, que contem os genes da contagem
# [1:10] seleciona os 10 primeiros para visualização

In [ ]:
rownames(airway)[11:20]

In [ ]:
# Estatísticas básicas
colSums(assay(airway))
# colSums() soma as contagens por amostra, mostrando a profundidade de sequenciamento.

In [ ]:
summary(rowSums(assay(airway)))
# summary() mostra estatísticas básicas das contagens totais por gene.

## 3. Criando objeto DESeq e rodando o pipeline

### O que estamos fazendo aqui

* **Criando o objeto DESeqDataSet:** estrutura central do DESeq2 que combina contagens e metadados.

* **Filtrando genes pouco expressos:** reduz ruído e melhora a robustez da análise.

* **Rodando o pipeline com `DESeq()`:** normalização, dispersão e testes estatísticos em um só passo.

* **Extraindo resultados com `results()`:** tabela de genes com log2FC e significância.

* **Inspecionando com `head()`:** primeira visão dos genes diferencialmente expressos.

In [ ]:
# Criar objeto DESeqDataSet
dds <- DESeqDataSet(airway, design = ~ cell + dex)

# Aqui transformamos o objeto airway em um DESeqDataSet.
# O argumento 'design = ~ cell + dex' define o modelo estatístico:
# queremos avaliar o efeito do tratamento com dexametasona (dex),
# levando em conta também o tipo celular (cell) como covariável.

# Filtrar genes pouco expressos
dds <- dds[rowSums(counts(dds)) > 1, ]

# Muitas vezes genes com contagens muito baixas não trazem informação útil
# e podem aumentar o ruído da análise.
# Aqui removemos genes cuja soma de contagens em todas as amostras é <= 1.

# Rodar pipeline completo
dds <- DESeq(dds)

# Este comando executa várias etapas automaticamente:
# - Normalização das contagens (size factors)
# - Estimativa da dispersão dos genes
# - Ajuste do modelo estatístico (regressão binomial negativa)
# - Teste de expressão diferencial (por padrão, Wald test)

In [ ]:
# Extrair resultados
res <- results(dds)
# 'results()' gera uma tabela com:
# - log2FoldChange: magnitude da diferença de expressão
# - pvalue: significância estatística
# - padj: p‑valor ajustado para múltiplos testes

# Visualizar primeiras linhas
head(res)
# 'head()' mostra apenas o topo da tabela de resultados,
# útil para inspecionar rapidamente os genes mais relevantes.

In [ ]:
# Extrair resultados especificando contraste
res <- results(dds, contrast = c("dex", "trt", "untrt"))
# Aqui usamos o argumento 'contrast' para definir a comparação desejada.
# A sintaxe é: contrast = c("variável", "nível_de_interesse", "nível_de_referência").
# No exemplo acima:
# - "dex" é a variável (condição experimental: tratado vs não tratado).
# - "trt" é o nível de interesse (tratamento com dexametasona).
# - "untrt" é o nível de referência (controle, sem tratamento).
# Assim, o log2FoldChange será calculado como (trt / untrt).

# Visualizar primeiras linhas
head(res)
# Mostra os genes com seus valores de log2FC, p‑valor e p‑valor ajustado.


`contrast()` permite escolher explicitamente qual condição comparar contra qual referência.

Isso é útil quando há múltiplos fatores no design (ex.: tipo celular, sexo, lote).

O resultado `(res)` muda dependendo do contraste escolhido, mostrando diferentes genes diferencialmente expressos.

In [ ]:
# Comparação entre tipos celulares
res_cell <- results(dds, contrast = c("cell", "N61311", "N052611"))
# Aqui comparamos o tipo celular N61311 contra N052611.

# Visualizar primeiras linhas
head(res_cell)


In [ ]:
# Comparação entre sexo (se estivesse no design)
res_sex <- results(dds, contrast = c("sexo", "M", "F"))
# Compararia amostras masculinas contra femininas.

# Visualizar primeiras linhas
head(res_sex)

Assim, vemos que não estamos limitados ao contraste padrão: podemos explorar diferentes comparações dentro do mesmo dataset.

## 4. Visualização dos Dados

PCA Plot (Análise de Componentes Principais).


O PCA ajuda a visualizar a similaridade global entre as amostras. Amostras do mesmo grupo devem se agrupar se o tratamento teve um efeito consistente.


**PCA básico por condição (tratamento vs controle)**

In [ ]:
# Transformação estabilizadora da variância
vsd <- vst(dds, blind = FALSE)

# A função vst() aplica uma transformação que estabiliza a variância das contagens.
# Isso é importante porque dados de RNA-seq têm variância dependente da média.
# Com a VST, os dados ficam mais adequados para visualizações como PCA e heatmaps.

# PCA simples usando função integrada
plotPCA(vsd, intgroup = "dex")

# Aqui o DESeq2 já gera um gráfico básico de PCA.
# intgroup = "dex" indica que queremos colorir os pontos pela condição (tratado vs controle).
# Esse gráfico mostra se as amostras se agrupam de acordo com o tratamento.


Variando as opções dentro do `plotPCA()`

**PCA por tipo celular**

In [ ]:
# PCA colorido por tipo celular
plotPCA(vsd, intgroup = "cell")
# Agora os pontos são coloridos pelo tipo celular, útil para ver se há efeito de batch ou origem.


**PCA com múltiplos fatores**

In [ ]:
# PCA com múltiplos fatores
plotPCA(vsd, intgroup = c("dex","cell"))
# Aqui o gráfico mostra cores e formas diferentes:
# - Cor representa condição (tratado vs controle)
# - Forma representa tipo celular

**PCA por sexo**

In [ ]:
plotPCA(vsd, intgroup = "sex")
# Se o metadado 'sex' estiver disponível, podemos colorir por sexo.
# Isso mostra se há diferenças globais de expressão associadas ao sexo das amostras.
# Mesmo que não seja o foco do experimento, é relevante explorar:
# pode indicar um fator biológico ou técnico que precisa ser considerado.


### Interpretação

* Agrupamento por condição (dex): indica efeito global do tratamento.

* Agrupamento por tipo celular (cell): mostra se há variabilidade associada ao tipo de célula.

* Combinação de fatores: ajuda a verificar se há interação ou confusão entre variáveis.

In [ ]:
# Converter resultados em data.frame
res_df <- as.data.frame(res)

# Criar coluna para indicar status de expressão diferencial
res_df$diffexpressed <- "Não significativo"

# Definição de limiares: |log2FoldChange| > 1 e padj < 0.05
res_df$diffexpressed[res_df$log2FoldChange > 1 & res_df$padj < 0.05] <- "Upregulado"
res_df$diffexpressed[res_df$log2FoldChange < -1 & res_df$padj < 0.05] <- "Downregulado"

# Aqui classificamos os genes em três categorias:
# - Upregulado: log2FC > 1 e padj < 0.05
# - Downregulado: log2FC < -1 e padj < 0.05
# - Não significativo: todos os demais


**Heatmap**

O heatmap mostra os genes mais variáveis e como eles se comportam em cada amostra.

Ele agrega informação porque:

* Permite ver padrões de co-expressão (genes que sobem ou descem juntos).

* Mostra se as amostras se agrupam por condição (tratamento vs controle) ou por outro fator (tipo celular, sexo, lote).

* Ajuda a detectar outliers: amostras que não seguem o padrão esperado.

* É uma forma visual de confirmar se os resultados estatísticos fazem sentido biologicamente.

In [ ]:
# Selecionar os genes mais variáveis
topVarGenes <- head(order(rowVars(assay(vsd)), decreasing = TRUE), 30)

# Criar heatmap
pheatmap(assay(vsd)[topVarGenes, ],
         cluster_rows = TRUE,
         cluster_cols = TRUE,
         annotation_col = as.data.frame(colData(vsd)[, c("dex","cell")]))
# Aqui selecionamos os 30 genes com maior variabilidade entre amostras.
# O heatmap mostra padrões de expressão desses genes.
# As colunas (amostras) são agrupadas por similaridade, e podemos ver se elas se separam por condição ou tipo celular.


**MA-plot**

O MA-plot mostra a relação entre a média de expressão (eixo X) e a mudança relativa (log2FC) (eixo Y).
Ele agrega informação porque:

* Permite ver se os genes diferencialmente expressos estão distribuídos em toda a faixa de expressão ou concentrados em genes de baixa/média expressão.

* Mostra se há tendência global (ex.: muitos genes upregulados ou downregulados).

* Destaca genes significativos em vermelho, facilitando a identificação visual.

* É útil para avaliar se os resultados são biologicamente plausíveis ou se há artefatos.

In [ ]:
plotMA(res, main="MA-plot DESeq2", ylim=c(-5,5))
# O MA-plot mostra a relação entre a média de expressão (eixo X)
# e o log2 Fold Change (eixo Y).
# Pontos em vermelho são genes significativamente diferencialmente expressos.


### PCA (Principal Component Analysis)

Mostra variabilidade global entre amostras: se as amostras se agrupam por condição (tratamento vs controle), tipo celular ou outros fatores.

Ajuda a identificar quais fatores estão influenciando os dados e se há separação clara entre grupos experimentais.

* É como olhar para o “mapa geral” da variabilidade.

### Heatmap

Padrões de variabilidade entre genes e amostras: quais genes são mais variáveis e como eles se comportam em cada amostra.

Revela padrões de co-expressão, agrupamento de amostras e possíveis outliers.

* É como olhar para os “atores principais” (genes) e ver como eles puxam a variabilidade.

### MA-plot
Revela a distribuição estatística das diferenças de expressão: relação entre média de expressão (intensidade) e magnitude da diferença (log2FC).

Destaca genes diferencialmente expressos e mostra se há tendência global (mais upregulados ou downregulados).

É como olhar para o “quanto” cada gene mudou e em que faixa de expressão.

### **Criação do Volcano Plot**

O Volcano Plot é uma visualização poderosa porque combina duas dimensões fundamentais da análise de expressão diferencial: a magnitude da mudança (log2 Fold Change) e a significância estatística (-log10 p‑valor ajustado).

In [ ]:
ggplot(res_df, aes(x = log2FoldChange, y = -log10(padj), color = diffexpressed)) +
    geom_point(alpha = 0.6, size = 1.5) +
    scale_color_manual(values = c("Downregulado" = "blue",
                                  "Não significativo" = "gray",
                                  "Upregulado" = "red")) +
    labs(title = "Volcano Plot: DEX vs CTRL",
         x = "Log2 Fold Change",
         y = "-Log10 p-valor ajustado") +
    theme_minimal() +
    geom_vline(xintercept = c(-1, 1), linetype = "dashed", color = "black") +
    geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "black")
# O Volcano Plot combina magnitude (log2FC) e significância (-log10 padj).
# Genes em vermelho: significativamente upregulados.
# Genes em azul: significativamente downregulados.
# Genes em cinza: não significativos.
# Linhas tracejadas indicam os limiares escolhidos (log2FC ±1 e padj 0.05).


### Interpretação

* Genes vermelhos (upregulados): aumentaram sua expressão após tratamento com dexametasona.

* Genes azuis (downregulados): tiveram expressão reduzida pelo tratamento.

* Genes cinza: não apresentaram alteração significativa.

* Eixo X (log2FC): magnitude da diferença de expressão.

* Eixo Y (-log10 padj): significância estatística (quanto mais alto, mais significativo).

* Linhas tracejadas: ajudam a visualizar os limiares escolhidos para definir relevância.

## 5. Enriquecimento Funcional

A lista de genes diferencialmente expressos por si só não diz muito.

Para entender o significado biológico, precisamos ver quais processos, vias ou funções estão associados a esses genes.

Esse processo é chamado de enriquecimento funcional: verificar se certos conjuntos de genes (ex.: vias metabólicas, processos celulares, funções moleculares) aparecem mais do que seria esperado por acaso.



## Bancos de dados usados
Existem vários bancos de dados que organizam genes em categorias biológicas:

**GO (Gene Ontology):** funções biológicas, moleculares e componentes celulares.

**KEGG:** vias metabólicas e de sinalização.

**Reactome:** processos biológicos detalhados.

**MSigDB:** coleções de assinaturas gênicas.

Cada banco traz uma perspectiva diferente sobre o que os genes estão fazendo.



### Ferramenta: clusterProfiler
No R, uma das ferramentas mais usadas é o clusterProfiler.

Ele permite rodar análises de enriquecimento em GO, KEGG e outros bancos.

Gera tabelas e gráficos (dotplots, barplots, enrichment maps) que mostram quais processos estão mais associados aos genes diferencialmente expressos.

É integrado com DESeq2: você pode passar diretamente os genes significativos e obter os resultados biológicos.

### Instalando e carregando os pacotes necessários para essa etapa

In [ ]:
# Instalar pacotes necessários
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

BiocManager::install(c("clusterProfiler", "org.Hs.eg.db", "enrichplot"))

# Carregar pacotes
library(clusterProfiler)
library(org.Hs.eg.db)   # Banco de anotação para Homo sapiens
library(enrichplot)     # Funções de visualização

**Preparação dos genes**

In [ ]:
# Selecionar genes significativos (padj < 0.05)
sig_genes <- rownames(res)[which(res$padj < 0.05 & !is.na(res$padj))]



In [ ]:
# Converter IDs de genes de ENSEMBL para ENTREZ (necessário para clusterProfiler)
sig_genes_entrez <- mapIds(org.Hs.eg.db,
                           keys = sig_genes,
                           column = "ENTREZID",
                           keytype = "ENSEMBL",
                           multiVals = "first")

# Remover genes sem correspondência
sig_genes_entrez <- na.omit(sig_genes_entrez)

**Enriquecimento Funcional**

In [ ]:
# Enriquecimento GO (Gene Ontology - Biological Process)
ego <- enrichGO(gene          = sig_genes_entrez,
                OrgDb         = org.Hs.eg.db,
                keyType       = "ENTREZID",
                ont           = "BP",       # BP = Biological Process
                pAdjustMethod = "BH",       # Correção de múltiplos testes
                pvalueCutoff  = 0.05,
                qvalueCutoff  = 0.05)

# Enriquecimento KEGG (vias metabólicas e de sinalização)
ekegg <- enrichKEGG(gene         = sig_genes_entrez,
                    organism     = 'hsa',   # hsa = Homo sapiens
                    pvalueCutoff = 0.05)


* GO (Gene Ontology): mostra funções biológicas gerais (ex.: resposta imune, apoptose).

* KEGG: mostra vias específicas de sinalização/metabolismo (ex.: NF-kappa B, glicocorticoides).

O enriquecimento testa se esses conjuntos de genes aparecem mais do que por acaso entre os genes diferencialmente expressos.

**Visualizações**

In [ ]:
# Dotplot GO
dotplot(ego, showCategory = 10) + ggtitle("GO Enrichment - Biological Processes")


In [ ]:
# Dotplot KEGG
dotplot(ekegg, showCategory = 10) + ggtitle("KEGG Pathways Enrichment")

In [ ]:
# Barplot como alternativa
barplot(ego, showCategory = 10, title = "GO Enrichment - Top Categories")

* **Dotplot:** mostra categorias enriquecidas como pontos, onde o tamanho indica número de genes e a cor indica significância.

* **Barplot:** alternativa que ordena os termos mais significativos em barras.

### Interpretação
O enriquecimento funcional traduz a lista de genes em significado biológico.

Se vemos termos como resposta inflamatória ou via de sinalização de glicocorticoides, isso confirma que o tratamento com dexametasona está modulando processos esperados.

O GO mostra funções gerais, enquanto o KEGG mostra vias específicas.

Juntos, eles revelam o impacto real do tratamento nas células.

### Salvar arquivos

In [ ]:
# Salvar resultados do DESeq2 em CSV
write.csv(as.data.frame(res), file = "DESeq2_results.csv")

# Salvar resultados de enriquecimento GO
write.csv(as.data.frame(ego), file = "GO_enrichment_results.csv")

# Salvar resultados de enriquecimento KEGG
write.csv(as.data.frame(ekegg), file = "KEGG_enrichment_results.csv")


### Exportar para o computador

In [ ]:
from google.colab import files

# Baixar os arquivos gerados
files.download("DESeq2_results.csv")
files.download("GO_enrichment_results.csv")
files.download("KEGG_enrichment_results.csv")


# Links úteis:

* Single Cell Learn é uma iniciativa para ensino de Single Cell RNA-Seq e Spatial Transcriptomics. Os módulos introdutórios podem ajudar vocês a começar a explorar datasets e usar o R
https://integrativebioinformatics.github.io/scNotebooks/modules/pt/modulo01/modulo01.html
https://integrativebioinformatics.github.io/scNotebooks/modules/pt/modulo02/modulo02.html

* Vignette do DESeq2 com informações em detalhes e mais conteúdos para serem explorados
https://bioconductor.org/packages//release/bioc/vignettes/DESeq2/inst/doc/DESeq2.html

* BioStar Handbook é um ótimo livro para ensinar a programação e a interpretar dados biológicos:
https://www.biostarhandbook.com/

* Curso da FutureLearn que abre periodicamente
https://coursesandconferences.wellcomeconnectingscience.org/event/bioinformatics-for-biologists-an-introduction-to-linux-bash-scripting-and-r-20260323/

* Repositório de ensino ao R
https://pedropark99.github.io/Introducao_R/

* Livro com tutorial ao R
https://epirhandbook.com/pt/new_pages/basics.pt.html